# Isothermal 0D Master Equation Simulation
### Import libraries

In [ ]:
import os
import sys
import importlib
import numpy as np
import pandas as pd
import tensorflow as tf

from tqdm.notebook import tqdm

%matplotlib notebook

Global paths

In [ ]:
# Current Working Directory
CWD = os.getcwd()
# WORKSPACE
WORKSPACE = os.environ['WORKSPACE_PATH']

In [ ]:
# PyCOMET
PyCOMET = WORKSPACE + '/ML/PyCOMET/pycomet/'
if importlib.util.find_spec('pycomet') is None:
    sys.path.append(os.path.abspath(PyCOMET))

In [ ]:
# SurQCT
SurQCT = WORKSPACE + '/ML/PyCOMET/pycomet/examples/SurQCT/'
sys.path.append(os.path.abspath(SurQCT))

from surqct.src.chemical_systems import systems as syst
from surqct.src.postprocessing.MasterEquation import Plotting

Case-specific paths

In [ ]:
# Output directory
out_dir = CWD + '/../Output/MasterEquation/em14/'
if not os.path.exists(out_dir):
    os.makedirs(out_dir)

### Initializing Runs

Global inputs

In [ ]:
SystemName   = 'O3_UMN'
# CaseSuffixes = {
#     'QCT':    '/em16/',
#     'SurQCT-C6.10': '/em16/configs.log10/config.06/run.03/',
#     'SurQCT-C6.e': '/em16/configs.ln/config.06/run.03/',
#     'SurQCT-C3.10': '/em16/configs.log10/config.03/run.03/',
#     'SurQCT-C3.e': '/em16/configs.ln/config.03/run.03/'
# }
TTranVec    = [1500, 5000, 10000, 20000]
# TTranVec    = [1500, 2500, 5000, 8000, 10000, 12000, 15000, 20000]

In [ ]:
# CaseSuffixes = {
#     'QCT':    '/mixed/',
#     'QCT-Corr': '/mixed.corr/'
#     # 'SurQCT': '/final/mixed.v2/'
# }
CaseSuffixes = {
    'QCT':        '/em14/',
    'SurQCT-Ref': '/em14.ref/',
    'SurQCT-C1':  '/em14.v1/',
    'SurQCT-C2':  '/em14.v2/'
}

#### Initialization

Initializing chemical systems

In [ ]:
Systems = {}
for KinSource in CaseSuffixes.keys():
    Systems[KinSource] = syst.get_system(
        NameLong=SystemName,
        KinSource=KinSource.split('-')[0]
    )

### Postprocessing Master Equation Solution

In [ ]:
# for KinSource, System in Systems.items():
#     System.MasterEquation.postprocess_data(CaseSuffix=CaseSuffixes[KinSource])
#     System.MasterEquation.save_data()

In [ ]:
for KinSource, System in Systems.items():
    System.MasterEquation.read_data(CaseSuffix=CaseSuffixes[KinSource])

### Plotting
Energies

In [ ]:
Data = {}
for KinSource, System in Systems.items():
    for ProcCode in System.MasterEquation.Data.keys():
        if ProcCode not in Data:
            Data[ProcCode] = {}
        Data[ProcCode][KinSource] = {}
        for TTran in TTranVec:
            Time     = System.MasterEquation.Data[ProcCode][str(TTran)]['t']
            Energies = System.MasterEquation.Data[ProcCode][str(TTran)]['Energies']
            Data[ProcCode][KinSource][TTran] = (Time, Energies)

In [ ]:
for ProcCode in Data.keys():
    Plotting.energies(out_dir+'/'+ProcCode, System, Data[ProcCode], TTranVec)

Taus

In [ ]:
Data = {}
for KinSource, System in Systems.items():
    for ProcCode in System.MasterEquation.Data.keys():
        if ProcCode.startswith('0'):
            if ProcCode not in Data:
                Data[ProcCode] = {}
            Data[ProcCode][KinSource] = System.MasterEquation.Data[ProcCode]['Taus']
Data = tf.nest.map_structure(pd.DataFrame.from_records, Data)
# Data = tf.nest.map_structure(lambda x: x[x['T'].isin(TTranVec)], Data)

In [ ]:
# for ProcCode in Data.keys():
#     Plotting.taus(out_dir+'/'+ProcCode, System, Data[ProcCode])

Mole fraction

In [ ]:
Data = {}
for KinSource, System in Systems.items():
    for ProcCode in System.MasterEquation.Data.keys():
        if not ProcCode.startswith('0'):
            if ProcCode not in Data:
                Data[ProcCode] = {}
            Data[ProcCode][KinSource] = {}
            for TTran in TTranVec:
                DSet = System.MasterEquation.Data[ProcCode][str(TTran)]
                X    = {k: v for k, v in DSet.items() if 'X_' in k}
                Data[ProcCode][KinSource][TTran] = (DSet['t'], X)

In [ ]:
for ProcCode in Data.keys():
    Plotting.mole_fraction(out_dir+'/'+ProcCode, System, Data[ProcCode], TTranVec)

Mole fraction depletion delays

In [ ]:
Data = {}
for KinSource, System in Systems.items():
    for ProcCode in System.MasterEquation.Data.keys():
        if not ProcCode.startswith('0'):
            if ProcCode not in Data:
                Data[ProcCode] = {}
            Data[ProcCode][KinSource] = {}
            for TTran in TTranVec:
                Data[ProcCode][KinSource][TTran] = System.MasterEquation.Data[ProcCode][str(TTran)]['MassDeplTime']

In [ ]:
# for ProcCode in Data.keys():
#     Plotting.mole_depletion_delays(out_dir+'/'+ProcCode, System, Data[ProcCode], TTranVec)

Thermal/QSS rates

In [ ]:
Data = {}
for KinSource, System in Systems.items():
    for ProcCode in System.MasterEquation.Data.keys():
        if not ProcCode.startswith('0'):
            if ProcCode not in Data:
                Data[ProcCode] = {}
            Data[ProcCode][KinSource] = System.MasterEquation.Data[ProcCode]['KGlobal']
Data = tf.nest.map_structure(pd.DataFrame.from_records, Data)
# Data = tf.nest.map_structure(lambda x: x[x['T'].isin(TTranVec)], Data)

In [ ]:
# for ProcCode in Data.keys():
#     Plotting.KD_ThQSS(out_dir+'/'+ProcCode, System, Data[ProcCode])